In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("LabTestRecommender") \
    .master("local[*]") \
    .config("spark.driver.memory","12g") \
    .config("spark.sql.shuffle.partitions","200") \
    .getOrCreate()

In [2]:
labs = spark.read.parquet("parquet/labevents")

labs = labs.select(
    "hadm_id",
    "itemid",
    "charttime"
).dropna()

In [3]:
labs = labs.repartition(200, "hadm_id")

In [4]:
window = Window.partitionBy("hadm_id").orderBy("charttime")

labs_seq = labs.withColumn(
    "prev_item",
    lag("itemid").over(window)
)

labs_seq = labs_seq.dropna()

In [7]:
window2 = Window.partitionBy("hadm_id").orderBy("charttime")

ranked = labs.withColumn(
    "rank",
    row_number().over(window2)
)

max_rank = ranked.groupBy("hadm_id").agg(
    max("rank").alias("max_rank")
)

ranked = ranked.join(max_rank, "hadm_id")

train = ranked.filter(col("rank") < col("max_rank"))
test = ranked.filter(col("rank") == col("max_rank"))

In [8]:
train_seq = train.withColumn(
    "prev_item",
    lag("itemid").over(window)
).dropna()

transition_counts = train_seq.groupBy(
    "prev_item","itemid"
).count()

In [9]:
total = transition_counts.groupBy(
    "prev_item"
).agg(sum("count").alias("total"))

markov = transition_counts.join(
    total,
    "prev_item"
).withColumn(
    "markov_prob",
    col("count")/col("total")
)

In [10]:
pairs = train.alias("a").join(
    train.alias("b"),
    "hadm_id"
).filter(
    col("a.itemid") != col("b.itemid")
)

co_counts = pairs.groupBy(
    col("a.itemid").alias("item_i"),
    col("b.itemid").alias("item_j")
).count()

In [11]:
total_i = co_counts.groupBy("item_i").agg(
    sum("count").alias("total")
)

co_matrix = co_counts.join(
    total_i,
    "item_i"
).withColumn(
    "co_prob",
    col("count")/col("total")
)

In [12]:
alpha = 0.7
combined = markov.join(
    co_matrix,
    (markov.prev_item == co_matrix.item_i) &
    (markov.itemid == co_matrix.item_j),
    "left"
).fillna(0)
combined = combined.withColumn(
    "score",
    alpha*col("markov_prob") + (1-alpha)*col("co_prob")
)

In [13]:
window_rank = Window.partitionBy("prev_item").orderBy(col("score").desc())

topk = combined.withColumn(
    "rank",
    row_number().over(window_rank)
).filter(col("rank") <= 3)

In [14]:
test_input = train_seq.groupBy("hadm_id").agg(
    max("itemid").alias("last_test")
)

test_label = test.select(
    "hadm_id",
    col("itemid").alias("true_next")
)

In [15]:
pred = test_input.join(
    topk,
    test_input.last_test == topk.prev_item
)

In [16]:
top1 = pred.filter(col("rank")==1)

acc = top1.join(
    test_label,
    "hadm_id"
).filter(
    col("itemid")==col("true_next")
).count() / test_label.count()

print("Accuracy@1 =", acc)

Accuracy@1 = 0.07225775035794937


In [ ]:
hits = pred.join(
    test_label,
    "hadm_id"
).filter(
    col("itemid")==col("true_next")
)

hitrate = hits.select("hadm_id").distinct().count() / test_label.count()

print("HitRate@3 =", hitrate)